# 02 — Train detector YOLO

    Chạy khi đã có dataset ZIP gán nhãn và cần so sánh detector với ROI theo camera. Mở runtime GPU. Điền URL Git, chạy các cell theo thứ tự. Checkpoint nằm trên Drive; ảnh train nằm ở `/content`. Với camera cố định, có thể bắt đầu reader trước rồi quay lại detector nếu ROI không ổn định.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
from google.colab import drive

REPO_URL = 'https://github.com/huybitvvt/scale-ocr.git'
PROJECT_COMMIT = ''  # Điền commit SHA cố định trước run chính; để trống chỉ khi thăm dò.
GITHUB_SECRET_NAME = ''  # Repo public không cần token; chỉ đặt tên secret nếu chuyển private.
VERSION = 'v001'
DRIVE_ROOT = Path('/content/drive/MyDrive/tram-can')
LOCAL_ROOT = Path('/content/scale-data')
REPO_ROOT = Path('/content/scale-ocr-code')
drive.mount('/content/drive')
assert DRIVE_ROOT.is_dir(), 'Không thấy MyDrive/tram-can: mount đúng tài khoản Google chứa ZIP'
assert REPO_URL.startswith(('https://', 'git@')), 'Điền REPO_URL sau khi push Git'
if not REPO_ROOT.exists():
    git_env = os.environ.copy()
    askpass = Path('/content/scale-ocr-askpass.sh')
    try:
        if GITHUB_SECRET_NAME:
            from google.colab import userdata
            assert REPO_URL.startswith('https://github.com/'), 'Secret này chỉ dùng với github.com'
            token = userdata.get(GITHUB_SECRET_NAME)
            assert token, 'Chưa có Colab Secret GITHUB_TOKEN hoặc chưa cấp quyền notebook đọc secret'
            git_env['SCALE_GIT_TOKEN'] = token
            git_env['GIT_TERMINAL_PROMPT'] = '0'
            git_env['GIT_ASKPASS'] = str(askpass)
            askpass.write_text(chr(10).join([
                '#!/bin/sh', 'case "$1" in',
                '  *Username*) printf "%s\n" "x-access-token" ;;',
                '  *) printf "%s\n" "$SCALE_GIT_TOKEN" ;;',
                'esac', ''
            ]), encoding='utf-8')
            askpass.chmod(0o700)
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)],
                       env=git_env, check=True)
    finally:
        git_env.pop('SCALE_GIT_TOKEN', None)
        askpass.unlink(missing_ok=True)
if PROJECT_COMMIT:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', PROJECT_COMMIT], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
print('Code commit:', PROJECT_COMMIT)
print('Disk GiB free:', round(shutil.disk_usage('/content').free / 2**30, 2))
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Dataset đã duyệt nằm trên Drive; vòng lặp train chỉ đọc từ /content.
bundle = DRIVE_ROOT / 'datasets' / VERSION / 'dataset.zip'
checksum_file = bundle.with_name(bundle.name + '.sha256')
assert bundle.is_file() and checksum_file.is_file(), 'Upload dataset.zip và .sha256 sau khi gán/duyệt nhãn'
expected = checksum_file.read_text(encoding='ascii').split()[0].lower()
assert len(expected) == 64
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
local_bundle = LOCAL_ROOT / bundle.name
def sha256_stream(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if not local_bundle.exists() or sha256_stream(local_bundle) != expected:
    shutil.copyfile(bundle, local_bundle)
actual = sha256_stream(local_bundle)
assert actual == expected, f'Dataset SHA-256 mismatch: {actual}'
prepared = LOCAL_ROOT / 'prepared' / VERSION
if not (prepared / 'dataset_card.json').is_file():
    prepared.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_bundle) as archive:
        for item in archive.infolist():
            target = (prepared / item.filename).resolve()
            assert target.is_relative_to(prepared.resolve()), 'Unsafe ZIP path'
        archive.extractall(prepared)
card = json.loads((prepared / 'dataset_card.json').read_text(encoding='utf-8'))
assert card['counts'].get('recognizer_crops_train', 0) > 0, 'Dataset thiếu crop train'
print('Dataset SHA-256:', actual)
print('Counts:', card['counts'])


In [ ]:
import torch, yaml
from datetime import datetime, timezone
assert torch.cuda.is_available(), 'Bật GPU trong Colab Runtime settings'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics'], check=True)
from ultralytics import YOLO
detector_root = prepared / 'detector'
config = yaml.safe_load((detector_root / 'data.yaml').read_text(encoding='utf-8'))
config['path'] = str(detector_root)
local_yaml = detector_root / 'data.colab.yaml'
local_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
assert (detector_root / 'images' / 'train').is_dir()
RUN_ID = 'det-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = DRIVE_ROOT / 'runs' / RUN_ID
assert not run_dir.exists(), 'Run ID đã tồn tại'
run_dir.mkdir(parents=True)
(run_dir / 'run_manifest.json').write_text(json.dumps({
    'project_commit': PROJECT_COMMIT, 'dataset_sha256': actual,
    'dataset_version': VERSION, 'model': 'yolo26s.pt',
    'yaml': config, 'seed': 42, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
}, ensure_ascii=False, indent=2), encoding='utf-8')
print('Run:', RUN_ID)


In [ ]:
# Thử 1 epoch bằng RUN_ID riêng trước run dài. Chỉ dùng ảnh train/val ở đây.
model = YOLO('yolo26s.pt')
model.train(data=str(local_yaml), imgsz=1024, epochs=100, batch=4,
    device=0, workers=2, optimizer='AdamW', lr0=0.001,
    patience=20, seed=42, fliplr=0, flipud=0, mosaic=0, mixup=0,
    degrees=5, translate=0.03, scale=0.1, hsv_h=0, hsv_s=0.2, hsv_v=0.2,
    project=str(DRIVE_ROOT / 'runs'), name=RUN_ID,
    save=True, save_period=5, exist_ok=True)


## Resume khi Colab ngắt

    Chạy lại bootstrap và copy **cùng version dataset** vào `/content`. Điền ID run cũ. Chỉ dùng `last.pt` để tiếp tục optimizer; `best.pt` dùng để chọn/suy luận.


In [ ]:
RESUME_RUN_ID = ''  # Chỉ điền khi cần resume; không chạy cell này cho run mới.
if RESUME_RUN_ID:
    old_run = DRIVE_ROOT / 'runs' / RESUME_RUN_ID
    old = json.loads((old_run / 'run_manifest.json').read_text(encoding='utf-8'))
    assert old['project_commit'] == PROJECT_COMMIT and old['dataset_sha256'] == actual
    last = old_run / 'weights' / 'last.pt'
    assert last.is_file(), 'Không có last.pt để resume'
    YOLO(str(last)).train(resume=True)
